# Lab 13: Hubs, Preferential Attachment, Heavy Tails

In this lab, you will investigate how networks can develop hubs.

Some graphs distribute degree fairly evenly. Others have a few vertices with very large degree and many vertices with small degree.

One possible mechanism for producing hubs is **preferential attachment**: new vertices are more likely to connect to vertices that are already well connected.

In this lab, you will:
- compare random graphs and preferential-attachment graphs,
- study degree sequences and degree histograms,
- look at cumulative degree plots,
- examine hub formation,
- compare observed structure to model-generated structure.

You should also keep a skeptical question in mind:

Does seeing hubs or a heavy tail automatically justify calling a network “scale-free”?

#### Note on datasets

At this point, we will pivot away from our 'toy' examples (path, star, cycle, karate club) and introduce a new dataset: the internet autonomous system.
The idea here is to generate a graph that has similar connectivity to internet systems.
The nodes represent autonomous systems and are *typed* (aka they have an attribute assigned to them), where the types represent tier-1 (T), mid-level (M), customer (C) or content-provider (CP). Each edge models a communication link (hence, bidirectional).
The full reference to this dataset and how it is generated can be found at: 
https://networkx.org/documentation/stable/reference/generated/networkx.generators.internet_as_graphs.random_internet_as_graph.html

#### Note on baseline models

In this lab, we'll use two baseline models for comparison:
- ER random graph - this will serve as our completely random baseline
- Barabasi-Albert graph model - this will serve as our preferential attachment baseline

In addition, we will have the IAS observed model described above. Since this is also a 'model' of sorts, we are adding an extra layer of 'degrees of freedom'. For example:
- in every network, we can choose a starting $n$ nodes
- for the ER networks, we can choose $p$ for the probability an edge is created
- for the BA networks, we can $m$, the number of edges to attach from a new node to existing nodes
- the random seed $rs$ and the number of simulations $sim$ what we wish to run for the null models 

Combined, these variables form the *parameters* for our experiments. In this case, we are dealing with a limited set of parameters, so things are still manageable from an experimental point of view. In real world settings, there may be many more 'levers' to pull; this is where *hyperparameter sweeps* come into play. Typically, in an experimental framework, we will set up a grid of options as a list (i.e. let's try this for $n \in [1000, 5000, 10000], p \in [0.3, 0.5, 0.7 ]$, etc.) and write a loop to sweep over all possible configurations. This is a way to build up more evidence to support a claim experimentally and can create data-driven hypotheses that lead to deeper mathematical insights.

In [ ]:
from lab13_helpers import *
import networkx as nx

### Part 1: Generate the networks

In [ ]:
# Build the observed network and our null models
# Let's define a single global variable for the number of nodes in the graph
# as mentioned above, this will help us run a sweep later on
N_NODES = 300
SEED = 13
TRIALS = 500

# Observed network
AS = internet_as_graph(300, seed=SEED)
N_EDGES = AS.number_of_edges()

# Null models
ER = er_graph(N_NODES, N_EDGES, SEED)
BA = ba_graph(N_NODES, 2, SEED)

# Generate an initial comparison of basic metrics
compare_basic_degree_stats([AS, ER, BA], ["Internet", "ER", "BA"])

Questions:
- Which graph has the largest maximum degree?
- Which graph has the lowest density?
- Which graph seems most likely to contain strong hubs?
- Why is it useful to compare graphs at roughly the same scale?

### Part 2: First visual comparison

Now compare the three graphs visually.

At first, do not worry about exact numbers. Just look for the presence or absence of hubs and local structure.

In [ ]:
draw_graph(ER, "ER graph", with_labels=False)
draw_graph(BA, "BA graph", with_labels=False)
draw_graph_by_type(AS, "Internet AS graph by node type")

In [ ]:
draw_graph_sized_by_degree(ER, "ER graph sized by degree")
draw_graph_sized_by_degree(BA, "BA graph sized by degree")
draw_graph_sized_by_degree(AS, "Internet AS graph sized by degree")

Questions:
- In which graph do hubs stand out most clearly?
- Which graph looks most homogeneous?
- Which graph looks most hierarchical?
- Does the Internet AS graph look more like the ER graph or the BA graph?

### Part 3: Degree sequences and top-degree vertices

Now compute the degree sequence and identify the highest-degree vertices.

In [ ]:
for net in [AS, ER, BA]:
    print(top_k_degrees(net))

In [ ]:
for net in [AS, ER, BA]:
    print(degree_sequence(net)[:20])

In [ ]:
plot_degree_histogram(ER, "ER graph degree histogram")
plot_degree_histogram(BA, "BA graph degree histogram")
plot_degree_histogram(AS, "Internet AS graph degree histogram")

Questions:
- Which graph has the most extreme top-degree vertices?
- Which graph has the most even degree sequence?
- Does the observed AS graph contain vertices whose degree is much larger than the rest?
- What does that suggest about hub structure?

### Part 4: Looking at the tail

For larger graphs, histograms can make it hard to see what is happening in the high-degree tail.

So we also examine cumulative and complementary cumulative views.

In [ ]:
plot_degree_cdf(ER, "ER graph degree CDF")
plot_degree_cdf(BA, "BA graph degree CDF")
plot_degree_cdf(AS, "Internet AS graph degree CDF")

In [ ]:
plot_degree_ccdf(ER, "ER graph degree CCDF")
plot_degree_ccdf(BA, "BA graph degree CCDF")
plot_degree_ccdf(AS, "Internet AS graph degree CCDF")

In [ ]:
plot_degree_ccdf_loglog(ER, "ER graph CCDF (log-log)")
plot_degree_ccdf_loglog(BA, "BA graph CCDF (log-log)")
plot_degree_ccdf_loglog(AS, "Internet AS graph CCDF (log-log)")

Questions:
- Which graph has the heaviest-looking tail?
- Which graph looks least compatible with a narrow degree range?
- Does the observed AS graph appear closer to the ER null or the BA null in the tail?
- Why might people use CCDF or log-log style views when studying hubs?

### Part 5: Node types and hierarchy in the observed network

The Internet AS graph model includes node types that reflect structural roles in the network.

These types help us think of the graph as more than just a degree sequence.

In [ ]:
type_counts(AS)

In [ ]:
node_type_table(AS).head(20)

Questions:
- What kinds of node roles appear in the observed network?
- Why might a technological network naturally produce hierarchy?
- Do hierarchy and hub structure seem related here?
- Why is this observed network more realistic than a toy star graph?

### Part 6: Direct comparison of observed graph to null models

Now compare the observed Internet AS graph directly to the two null models.

The point is not to prove that one model generated the observed graph, but to ask which null captures the observed degree structure more plausibly.

In [ ]:
compare_basic_degree_stats([AS, ER, BA], ["Observed AS", "ER null", "BA null"])

Question:
- from this table alone, is it obvious which null serves as a better baseline?

### Part 7: Statistical testing against the ER and BA null models

So far, we have compared the observed Internet AS graph to the ER and BA models descriptively.

Now we turn that comparison into statistical testing.

We will use two hub-sensitive statistics:

- maximum degree,
- degree Gini coefficient.

The first asks how large the largest hub is.
The second asks how unequal the degree distribution is across the whole graph.

In [ ]:
observed_stats = observed_hub_statistics(AS)
observed_stats

For the ER null model, we test:

$\mathcal{H}_0^{ER}$: The observed AS graph is consistent with an ER graph of the same size.

$\mathcal{H}_1^{ER}$: The observed AS graph has unusually strong hub concentration relative to the ER null.

For the BA null model, we test:

$\mathcal{H}_0^{BA}$: The observed AS graph is consistent with a BA graph of the same size and similar average degree.

$\mathcal{H}_1^{BA}$: The observed AS graph has unusually strong hub concentration relative to the BA null.

In [ ]:
n = AS.number_of_nodes()
m_edges = AS.number_of_edges()

m_ba = max(1, round(m_edges / n))
m_ba

The BA parameter $m$ is chosen so that the BA graph has roughly the same average degree as the observed graph.

In [ ]:
er_null = simulate_er_hub_statistics(n, m_edges, trials=TRIALS, seed=SEED)
ba_null = simulate_ba_hub_statistics(n, m_ba, trials=TRIALS, seed=SEED)

### Part 7A: Testing the maximum degree

The first statistic is the maximum degree.

A large maximum degree suggests the presence of a strong hub.

In [ ]:
observed_stats["max_degree"]

In [ ]:
summarize_null(er_null["max_degree"])

In [ ]:
summarize_null(ba_null["max_degree"])

In [ ]:
plot_null_histogram(
    er_null["max_degree"],
    observed_value=observed_stats["max_degree"],
    xlabel="Maximum degree",
    title="ER null distribution of maximum degree"
)

In [ ]:
plot_null_histogram(
    ba_null["max_degree"],
    observed_value=observed_stats["max_degree"],
    xlabel="Maximum degree",
    title="BA null distribution of maximum degree"
)

In [ ]:
p_er_max = empirical_p_value_upper(er_null["max_degree"], observed_stats["max_degree"])
p_ba_max = empirical_p_value_upper(ba_null["max_degree"], observed_stats["max_degree"])

p_er_max, p_ba_max

Questions:
- Is the observed maximum degree unusually large relative to the ER null?
- Is it unusually large relative to the BA null?
- Which null model gives the smaller $p$-value?
- What does that suggest about how well each null explains the largest hub?

### Part 7B: Testing degree inequality

The second statistic is the degree Gini coefficient.

This measures how unevenly degree is distributed across the vertices.
A larger value means more degree inequality.

In [ ]:
observed_stats["degree_gini"]

In [ ]:
summarize_null(er_null["degree_gini"])

In [ ]:
summarize_null(ba_null["degree_gini"])

In [ ]:
plot_null_histogram(
    er_null["degree_gini"],
    observed_value=observed_stats["degree_gini"],
    xlabel="Degree Gini coefficient",
    title="ER null distribution of degree inequality"
)

In [ ]:
plot_null_histogram(
    ba_null["degree_gini"],
    observed_value=observed_stats["degree_gini"],
    xlabel="Degree Gini coefficient",
    title="BA null distribution of degree inequality"
)

In [ ]:
p_er_gini = empirical_p_value_upper(er_null["degree_gini"], observed_stats["degree_gini"])
p_ba_gini = empirical_p_value_upper(ba_null["degree_gini"], observed_stats["degree_gini"])

p_er_gini, p_ba_gini

Questions:
- Is the observed degree inequality unusually large relative to the ER null?
- Is it unusually large relative to the BA null?
- Does the BA null explain the observed inequality better than the ER null?
- Are the conclusions from the Gini test consistent with the conclusions from the max-degree test?

### Part 7C: Interpreting the tests

Now compare the $p$-values across the two null models and the two test statistics.

The point is not to prove that the BA model is the true mechanism, but to ask which null gives a more plausible explanation of the observed hub structure.

In [ ]:
pd.DataFrame({
    "statistic": ["max_degree", "degree_gini"],
    "p_value_ER": [p_er_max, p_er_gini],
    "p_value_BA": [p_ba_max, p_ba_gini],
})

Questions:
- Which null model is harder to reject?
- Does that mean the BA model is a better descriptive model for hub structure?
- Does it prove preferential attachment generated the observed graph?
- Why is “better null fit” weaker than “identified mechanism”?